In [6]:
from ultralytics import YOLO
import os
import numpy as np
import cv2

# Count of Images in the dataset

In [3]:
dataset_path = "./archive"

all_images = []

for root, dirs, files in os.walk(dataset_path):

    for file in files:

        if file.endswith(".jpg") or file.endswith(".png"):

            all_images.append(os.path.join(root,file))

print("Total images:",len(all_images))

Total images: 1391


In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 2050


# Average size of the images

In [7]:
widths = []
heights = []

for img_path in all_images:

    img = cv2.imread(img_path)

    if img is not None:

        h,w,_ = img.shape

        widths.append(w)
        heights.append(h)

print("Average width:",np.mean(widths))
print("Average height:",np.mean(heights))
print("Min width:",min(widths))
print("Max width:",max(widths))

Average width: 666.0524802300504
Average height: 579.7958303378864
Min width: 133
Max width: 4000


# Convert dataset into YOLO format

## 1) create a directories for yolo model

In [31]:
import os

os.makedirs("dataset_new/images/train", exist_ok=True)
os.makedirs("dataset_new/images/val", exist_ok=True)
os.makedirs("dataset_new/labels/train", exist_ok=True)
os.makedirs("dataset_new/labels/val", exist_ok=True)

## 2) Define a Xml To Yolo function for convert a xml file to class labels

In [34]:
import xml.etree.ElementTree as ET

classes = ["license_plate"]   # your class names


def convert_xml_to_yolo(xml_path, img_w, img_h):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    labels = []

    for obj in root.iter("object"):

        class_name = obj.find("name").text

        if class_name not in classes:
            continue

        class_id = classes.index(class_name)

        xmlbox = obj.find("bndbox")

        xmin = float(xmlbox.find("xmin").text)
        ymin = float(xmlbox.find("ymin").text)
        xmax = float(xmlbox.find("xmax").text)
        ymax = float(xmlbox.find("ymax").text)

        # Convert to YOLO format
        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h

        label = f"{class_id} {x_center} {y_center} {width} {height}"

        labels.append(label)

    return labels

## 3) Convert DataSet into valid YOLO format (final phase)

In [ ]:
import shutil
import cv2
import random
import os

for img_path in all_images:

    xml_path = img_path.replace(".jpg",".xml").replace(".png",".xml")

    #  if xml not present
    if not os.path.exists(xml_path):
        continue

    img = cv2.imread(img_path)

    if img is None:
        continue

    h,w,_ = img.shape

    try:
        labels = convert_xml_to_yolo(xml_path, w
                                     
                                     
                                     , h)
    except:
        print("Skipping corrupted XML:", xml_path)
        continue

    file_name = os.path.basename(img_path)
    label_name = file_name.replace(".jpg",".txt").replace(".png",".txt")

    # train  split
    if random.random() < 0.8:
        img_dest = os.path.join("dataset_new/images/train", file_name)
        label_dest = os.path.join("dataset_new/labels/train", label_name)
    else:
        img_dest = os.path.join("dataset_new/images/val", file_name)
        label_dest = os.path.join("dataset_new/labels/val", label_name)

    shutil.copy(img_path, img_dest)

    with open(label_dest,"w") as f:
        f.write("\n".join(labels))

# Model Learning

### batbone layer - feature extraction
### neck - feature fusion (combine a feature)
### head - predict 

In [16]:

model = YOLO("yolov8n.pt")

model.train(
    data="license_plate.yaml", epochs=10, imgsz=416, batch=32, workers=0,device=0
)

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=license_plate.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train9, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x00000195956E3FD0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

# Test the result of model

In [17]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("runs/detect/train9/weights/best.pt")

results = model("random.avif")

result_img = results[0].plot()

plt.imshow(result_img)
plt.show()


image 1/1 C:\Users\yashp\Downloads\random.avif: 416x416 1 license_plate, 43.0ms
Speed: 5.9ms preprocess, 43.0ms inference, 6.0ms postprocess per image at shape (1, 3, 416, 416)


<Figure size 640x480 with 1 Axes>

In [18]:
import cv2
import matplotlib.pyplot as plt
img = cv2.imread("random.avif")

results = model(img)

for r in results:

    boxes = r.boxes.xyxy.cpu().numpy()

    for box in boxes:

        x1,y1,x2,y2 = map(int,box)

        plate = img[y1:y2, x1:x2]

        plt.imshow(cv2.cvtColor(plate, cv2.COLOR_BGR2RGB))
        plt.axis("off")
        plt.show()


0: 416x416 1 license_plate, 45.8ms
Speed: 7.5ms preprocess, 45.8ms inference, 9.3ms postprocess per image at shape (1, 3, 416, 416)


<Figure size 640x480 with 1 Axes>

In [19]:
import easyocr

reader = easyocr.Reader(['en'])

result = reader.readtext(plate)

for r in result:
    print(r[1])

TN 09 BY 9726


In [20]:
from ultralytics import YOLO

model = YOLO("runs/detect/train8/weights/best.pt")

metrics = model.val(data="license_plate.yaml")

Ultralytics 8.4.21  Python-3.10.19 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 2050, 4096MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 55.419.3 MB/s, size: 16.6 KB)
val: Scanning C:\Users\yashp\Downloads\dataset\labels\val.cache... 24 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 24/24  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 8.1s/it 16.3s<53.2s
                   all         24         24      0.955          1      0.992      0.755
Speed: 7.3ms preprocess, 21.9ms inference, 0.0ms loss, 5.9ms postprocess per image
Results saved to C:\Users\yashp\Downloads\runs\detect\val3


In [21]:
matrics

NameError: name 'matrics' is not defined